In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\nsf_valence'
Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings','audio')
df_ratings = pd.read_csv(os.path.join(Dir_ratings,'ratings_audio_att2_comp2.csv'), dtype={'idx': str,'ethnicity':str},keep_default_na=False)


In [4]:
df_ratings.loc[df_ratings['segID']==1].groupby(['group']).count()

,subID,idx,audio,audio_idx,segID,V,A,RT,age,sex,age_prescreen,sex_prescreen,attention_correct,attention_correct%,comp_correct,comp_correct%,source,ethnicity,residence_states
group,,,,,,,,,,,,,,,,,,,
middle,721,721,721,721,721,721,721,721,721,721,721,721,721,721,721,721,721,721,721
old,707,707,707,707,707,707,707,707,707,707,707,707,707,707,707,707,707,707,707
young,804,804,804,804,804,804,804,804,804,804,804,804,804,804,804,804,804,804,804


In [5]:
df_ratings_bysub = df_ratings.loc[df_ratings['segID']==1].groupby(['subID','group']).count().reset_index()
print(f"On average, each participant rated {df_ratings_bysub['audio'].mean().round(2)} narratives, with a range of [{df_ratings_bysub['audio'].min()},{df_ratings_bysub['audio'].max()}].")
df_ratings_bysub.groupby(['group']).mean(numeric_only=True).round(2)['audio']

On average, each participant rated 4.22 narratives, with a range of [1,18].


group
middle    4.51
old       7.07
young     2.99
Name: audio, dtype: float64

In [6]:
#### summarize ethnicity
df_demographic = df_ratings.drop_duplicates(subset=['subID'],keep='first')

print(f"The final sample consisted of {df_demographic.shape[0]} participants.")
for group in ['young','middle','old']:
    df = df_demographic.loc[df_demographic['group']==group].copy()
    print(f"----- {group} N = {df.shape[0]} -----")
    print(f"Among {group} adults, the mean age was {df['age'].mean().round(2)} years (SD = {np.round(df['age'].std(),2)}),")
    print(f"with {df.loc[df['sex']=='female'].shape[0]} females and {df.loc[df['sex']=='male'].shape[0]} males.")
    print('Ethnicity distribution was as follows: ')
    for ethnicity in ['White','Black','Asian','Other','NA']:
        
        N_group = df.shape[0]
        n_ethnicity = df.loc[df_ratings['ethnicity']==ethnicity].shape[0]
        # print(f"{n_ethnicity} ({np.round(n_ethnicity/N_group*100,2)}%) as {ethnicity}")
        print(f"{ethnicity} (n = {n_ethnicity}, {np.round(n_ethnicity/N_group*100,2)}%),")


The final sample consisted of 529 participants.
----- young N = 269 -----
Among young adults, the mean age was 22.92 years (SD = 5.67),
with 196 females and 73 males.
Ethnicity distribution was as follows: 
White (n = 167, 62.08%),
Black (n = 39, 14.5%),
Asian (n = 5, 1.86%),
Other (n = 23, 8.55%),
NA (n = 35, 13.01%),
----- middle N = 160 -----
Among middle adults, the mean age was 48.58 years (SD = 5.37),
with 87 females and 73 males.
Ethnicity distribution was as follows: 
White (n = 90, 56.25%),
Black (n = 48, 30.0%),
Asian (n = 3, 1.88%),
Other (n = 7, 4.38%),
NA (n = 12, 7.5%),
----- old N = 100 -----
Among old adults, the mean age was 65.71 years (SD = 4.35),
with 75 females and 25 males.
Ethnicity distribution was as follows: 
White (n = 69, 69.0%),
Black (n = 28, 28.0%),
Asian (n = 0, 0.0%),
Other (n = 2, 2.0%),
NA (n = 1, 1.0%),


#### Calculate the segment-level mean ratings by group and across groups

In [7]:
df_ratings_groupmean = df_ratings.groupby(['group','audio','segID']).mean(numeric_only=True).reset_index()
df_ratings_grandmean = df_ratings.groupby(['audio','segID']).mean(numeric_only=True).reset_index()
df_ratings_grandmean['group'] = 'all'
df_ratings_mean = pd.concat([df_ratings_groupmean,df_ratings_grandmean],ignore_index=True)
df_ratings_mean

,group,audio,segID,audio_idx,V,A,RT,age,attention_correct,attention_correct%,comp_correct,comp_correct%
0,middle,Allison_AgedMother,1,1.8125,0.690234,4.096875,2.722875,47.000000,2.875,0.958333,4.312500,0.862500
1,middle,Allison_AgedMother,2,1.8125,0.732422,4.671094,4.028375,47.000000,2.875,0.958333,4.312500,0.862500
2,middle,Allison_AgedMother,3,1.8125,-0.626953,4.687500,5.826187,47.000000,2.875,0.958333,4.312500,0.862500
3,middle,Allison_AgedMother,4,1.8125,1.219922,4.989844,2.900375,47.000000,2.875,0.958333,4.312500,0.862500
4,middle,Allison_AgedMother,5,1.8125,1.647656,4.800000,2.946500,47.000000,2.875,0.958333,4.312500,0.862500
...,...,...,...,...,...,...,...,...,...,...,...,...
24083,all,WavenetF_Koko,90,2.0000,2.058824,5.772794,4.110471,45.019608,3.000,1.000000,3.647059,0.911765
24084,all,WavenetF_Koko,91,2.0000,-0.021324,4.568750,2.124275,45.019608,3.000,1.000000,3.647059,0.911765
24085,all,WavenetF_Koko,92,2.0000,0.343750,5.459559,2.861667,45.019608,3.000,1.000000,3.647059,0.911765
24086,all,WavenetF_Koko,93,2.0000,0.289706,4.725735,2.200000,45.019608,3.000,1.000000,3.647059,0.911765


In [8]:
### add the segment texts and timestamps in the audio recordings
Dir_timestamps = os.path.join(Dir_github,'derivatives','behavioral','timestamps_segment')

dfs_timestamps = []
for audio in df_ratings_mean['audio'].unique():
    path_file = os.path.join(Dir_timestamps,f'timestamps_{audio}.csv')
    df_timestamps = pd.read_csv(path_file)
    dfs_timestamps.append(df_timestamps)
dfs_timestamps = pd.concat(dfs_timestamps,ignore_index=True)

df_ratings_mean = pd.merge(left = df_ratings_mean, left_on = ['audio','segID'],
         right = dfs_timestamps, right_on = ['voice_narrative','segID'],
         how = 'left')

df_ratings_mean.to_csv(os.path.join(Dir_github,'derivatives','behavioral','ratings','audio','ratings_audio_att2_comp2_mean.csv'),index=False)